# 2.2.0 - Kỹ thuật tạo đặc trưng (Feature Engineering)

Notebook này xây dựng **16 đặc trưng dự báo (Features)** cho mô hình theo 6 nhóm:

### 1. Lagged Returns (Trễ lợi suất)
$$\text{Return\_lag\_}k(t) = \text{Return}_{t-k}, \quad k \in \{1, 5, 20\}$$

### 2. Momentum (Đà tăng trưởng tích lũy)
$$\text{Momentum}_{kD}(t) = \sum_{i=0}^{k-1} \text{Return}_{t-i}, \quad k \in \{5, 20, 60\}$$

### 3. Rolling Volatility (Biến động lăn)
$$\text{Volatility}_{kD}(t) = \text{Std}(\text{Return}_{t-k+1}, \dots, \text{Return}_t), \quad k \in \{5, 20, 60\}$$

### 4. VIX-based Features (Đặc trưng tâm lý / biến động thị trường)
- **VIX level:** $\text{VIX}_t = \text{VIXCLS}_t$
- **VIX lag:** $\text{VIX\_lag\_1}(t) = \text{VIX}_{t-1}$
- **VIX absolute change:** $\text{VIX\_change\_1D}(t) = \text{VIX}_t - \text{VIX}_{t-1}$
- **VIX percentage change:** $\text{VIX\_pct\_change\_1D}(t) = \frac{\text{VIX}_t - \text{VIX}_{t-1}}{\text{VIX}_{t-1}}$

### 5. Price-based Features (Cấu trúc giá trong phiên)
- **High-Low range:** $\text{HL\_range}_t = \frac{\text{High}_t - \text{Low}_t}{\text{Close}_t}$
- **Open-Close return:** $\text{OC\_return}_t = \frac{\text{Close}_t - \text{Open}_t}{\text{Open}_t}$

### 6. Volume-based Feature (Khối lượng giao dịch)
- **Volume change:** $\text{Volume\_change}_t = \frac{\text{Volume}_t - \text{Volume}_{t-1}}{\text{Volume}_{t-1}}$

---
- **Dữ liệu đầu vào:** `dataset/2.1.0_target.csv`
- **Dữ liệu đầu ra:** `dataset/2.2.0_feature.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Định vị đường dẫn và đọc dữ liệu phiên bản 2.1.0_target.csv
dataset_dir = 'dataset' if os.path.exists('dataset') else ('../dataset' if os.path.exists('../dataset') else '.')
input_path = os.path.join(dataset_dir, '2.1.0_target.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Xây dựng 6 nhóm đặc trưng (Feature Engineering)

# Nhóm 1: Lagged Returns (Return_lag_k = Return_{t-k})
for k in [1, 5, 20]:
    df[f'Return_lag_{k}'] = df['Return_1D'].shift(k)

# Nhóm 2: Momentum (Tổng k ngày Return gần nhất)
for k in [5, 20, 60]:
    df[f'Momentum_{k}D'] = df['Return_1D'].rolling(window=k).sum()

# Nhóm 3: Rolling Volatility (Độ lệch chuẩn k ngày Return)
for k in [5, 20, 60]:
    df[f'Volatility_{k}D'] = df['Return_1D'].rolling(window=k).std()

# Nhóm 4: VIX-based Features
df['VIX_lag_1'] = df['VIXCLS'].shift(1)
df['VIX_change_1D'] = df['VIXCLS'].diff(1)
df['VIX_pct_change_1D'] = df['VIXCLS'].pct_change(1)

# Nhóm 5: Price-based Features
df['HL_range'] = (df['High'] - df['Low']) / df['Close']
df['OC_return'] = (df['Close'] - df['Open']) / df['Open']

# Nhóm 6: Volume-based Feature
df['Volume_change'] = df['Volume'].pct_change(1)

print("✅ Đã tạo xong toàn bộ các nhóm đặc trưng!")

In [ ]:
# 3. Thống kê và kiểm tra số lượng cột
feature_cols = [
    'Return_lag_1', 'Return_lag_5', 'Return_lag_20',
    'Momentum_5D', 'Momentum_20D', 'Momentum_60D',
    'Volatility_5D', 'Volatility_20D', 'Volatility_60D',
    'VIXCLS', 'VIX_lag_1', 'VIX_change_1D', 'VIX_pct_change_1D',
    'HL_range', 'OC_return',
    'Volume_change'
]

print(f"Tổng số cột trong Dataset: {len(df.columns)} cột")
print(f"Trong đó có {len(feature_cols)} Feature Columns phục vụ mô hình hóa:\n")
for idx, col in enumerate(feature_cols, 1):
    print(f"  {idx:2d}. {col}")

print("\n5 dòng đầu tiên của các features mới:")
display(df[['Date'] + feature_cols].head())

In [ ]:
# 4. Xuất file kết quả phiên bản 2.2.0_feature.csv
output_path = os.path.join(dataset_dir, '2.2.0_feature.csv')
df.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG TẠI: {output_path}")
print(f"Kích thước Dataset mới: {df.shape[0]} dòng, {df.shape[1]} cột.")
print("Danh sách toàn bộ các cột trong file:")
print(list(df.columns))